# GPT Actions - Snowflake 中间件
## 简介


本页面为针对特定应用构建 GPT Action 的开发者提供说明和指南。在继续之前，请先熟悉以下内容：

* [GPT Actions 简介](https://platform.openai.com/docs/actions)
* [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
* [从零构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)


本指南详细介绍如何将 ChatGPT 与 Snowflake 数据仓库连接，以便将 SQL 查询结果返回给 ChatGPT，用于[数据分析](https://help.openai.com/en/articles/8437071-data-analysis-with-chatgpt)。该 GPT 需要一个与中间件（例如 Azure function）交互的 action，使该 action 能够正确格式化来自 Snowflake 的响应，以便在 Python notebook 环境中使用。数据必须[以文件形式返回](https://platform.openai.com/docs/actions/sending-files/returning-files)，因此中间件函数应将 SQL 响应转换为大小不超过 10MB 的 CSV/Excel 文件。

本文档将概述这个中间件函数 GPT action。关于中间件函数本身的设置，请参阅 [GPT Actions 库（中间件）- Azure Functions](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_middleware_azure_function)。你可以将这个 Snowflake 中间件 action 与一个直连 Snowflake 的 action 结合使用，从而让 GPT 能够在执行 SQL 查询之前先构造并测试查询。

### 价值 + 业务用例示例


现有的 Snowflake 客户可以利用这些指南从数据仓库中查询数据，并将数据加载到数据分析的 Python 环境中以获得进一步的洞察。这使得由 ChatGPT 驱动的分析成为可能，例如可视化数据集、识别模式/异常，或为数据清洗找出缺口。这个 GPT 可用于基于相对较小的数据集推动业务决策，也可以在你用 BI 工具探索整体数据集时，借助 AI 探索数据子集以生成假设，从而节省时间和成本，同时发现以前未曾注意到的模式。

## 应用信息

### 应用关键链接


开始之前，请先查看 Snowflake 和 Azure 的以下链接：

**Snowflake Action**

* 应用网站：[https://app.snowflake.com/](https://app.snowflake.com/)
* 应用 Python Connector 文档：[https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-connect](https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-connect)

**Azure Function**

* 应用网站：[https://learn.microsoft.com/en-us/azure/azure-functions/](https://learn.microsoft.com/en-us/azure/azure-functions/)
* 应用 API 文档：[https://learn.microsoft.com/en-us/azure/azure-functions/functions-reference/](https://learn.microsoft.com/en-us/azure/azure-functions/functions-reference/)


### 应用前提条件


开始之前，请确保在你的应用环境中完成以下步骤：

* 预配一个 Snowflake 数据仓库
* 确保通过 ChatGPT 向 Snowflake 进行身份验证的用户，以所需的角色拥有对相应数据库、schema 和表的访问权限

此外，在 Azure Function App 中创建应用之前，你需要一种处理用户身份验证的方式。你需要在 Azure Entra ID 中设置一个 OAuth 应用注册，并将其与 Snowflake External OAuth 安全集成关联起来。Snowflake 的 External OAuth 安全集成允许外部系统签发访问令牌，Snowflake 可以据此确定访问级别。在这里，这个外部令牌提供方就是 Azure Entra ID。由于 ChatGPT 将连接到 Azure 而不是 Snowflake，GPT 用户的 OAuth 令牌将由 Azure 根据其在 Entra ID 中的用户身份进行签发。因此，你需要一种方式将 Snowflake 中的用户映射到 Azure 中对应的用户。 

Azure 端和 Snowflake 端所需的全部步骤如下所述。

## 在 Azure Entra ID 中配置 OAuth 资源


我们将设置一个新的应用注册，在 Azure 中配置将要使用的 Snowflake 作用域（Scopes），并获取 Snowflake 和 ChatGPT 中都需要用到的所有 OAuth 配置参数。本节的操作全部在 Azure 中进行，这样在下一节配置 Snowflake 端时，你就拥有了关联到此应用注册所需的信息。

1. 访问 [Microsoft Azure 门户](https://portal.azure.com/) 并完成身份验证。
2. 进入 Azure Entra ID（前身为 Active Directory）。
3. 点击 **Manage** 下的 **App Registrations**。
4. 点击 **New Registration**。
5. 在 **Name** 中输入 `Snowflake GPT OAuth Client` 或类似的值。
6. 确认 **Supported account types** 设置为 **Single Tenant**。
7. 暂时忽略 Redirect URI。配置 GPT 时你会再回来设置它
8. 点击 **Register**。
9. 记下 **Essentials** 下的 **Directory (tenant) ID**（`TENANT_ID`）。你将用它来生成 `AZURE_AD_ISSUER` 和 `AZURE_AD_JWS_KEY_ENDPOINT.`
    * `AZURE_AD_ISSUER` 为 `https://sts.windows.net/TENANT_ID/`
    *  `AZURE_AD_JWS_KEY_ENDPOINT` 为 `https://login.microsoftonline.com/TENANT_ID/discovery/v2.0/keys`
10. 在 **Overview** 界面中点击 **Endpoints**。
11. 在右侧，将 **OAuth 2.0 authorization endpoint (v2)** 记为 `AZURE_AD_OAUTH_AUTHORIZATION_ENDPOINT`，将 **OAuth 2.0 token endpoint (v2)** 记为 `AZURE_AD_OAUTH_TOKEN_ENDPOINT`。
    * 这些端点应类似于 `https://login.microsoftonline.com/90288a9b-97df-4c6d-b025-95713f21cef9/oauth2/v2.0/authorization` 和 `https://login.microsoftonline.com/90288a9b-97df-4c6d-b025-95713f21cef9/oauth2/v2.0/token`.
12. 点击 **Manage** 下的 **Expose an API **。
13. 点击 **Application ID URI** 旁边的 **Set** 链接，设置 `Application ID URI`。
    * `Application ID URI` 在你所在组织的目录中必须是唯一的，例如 `https://your.company.com/4d2a8c2b-a5f4-4b86-93ca-294185f45f2e`. 在后续配置步骤中，该值将被称为 `<SNOWFLAKE_APPLICATION_ID_URI>`。
14. 对于由程序化客户端代表用户执行操作的 OAuth 流程，要将某个 Snowflake 角色添加为 OAuth 作用域，请点击 **Add a scope**，添加一个代表该 Snowflake 角色的作用域。
    * 输入作用域时，在 Snowflake 角色名称前加上 `session:scope:` 前缀。例如，对于 Snowflake Analyst 角色，输入 `session:scope:analyst`。
    * 选择谁可以授予同意。
    * 为该作用域输入一个 **display name**（例如：Account Admin）。
    * 为该作用域输入一段 **description**（例如：Can administer the Snowflake account）。
    * 点击 **Add Scope**。
    * 将该作用域保存为 `AZURE_AD_SCOPE`。它应由你的 `Application ID URI` 和 `Scope name` 拼接而成
15. 在 **Overview** 部分，从 **Application (client) ID** 字段中复制 `ClientID`。在后续步骤中它将被称为 `OAUTH_CLIENT_ID`。
16. 点击 **Certificates & secrets**，然后点击 **New client secret**。
17. 为该密钥添加描述。
18. 选择 **730 days (24 months)**。出于测试目的，请选择不会很快过期的密钥。
19. 点击 **Add**。复制该密钥。在后续步骤中它将被称为 `OAUTH_CLIENT_SECRET`。
20. 对于将代表用户请求访问令牌的程序化客户端，按如下方式为应用配置委托权限（Delegated permissions）。
    * 点击 **API Permissions**。
    * 点击 **Add Permission**。
    * 点击 **My APIs**。
    * 点击你在[在 Azure AD 中配置 OAuth 资源](https://docs.snowflake.com/en/user-guide/oauth-azure#configure-the-oauth-resource-in-azure-ad)中创建的 **Snowflake OAuth Resource**。
    * 点击 **Delegated Permissions** 选项框。
    * 勾选与应用中定义的、你希望授予此客户端的作用域相关的权限。
    * 点击 **Add Permissions**。
    * 点击 **Grant Admin Consent** 按钮，向客户端授予这些权限。注意，这里以这种方式配置权限仅出于测试目的。在生产环境中，不建议以这种方式授予权限。
    * 点击 **Yes**。


## 在 Snowflake 中创建安全集成

在 Azure Entra ID 中完成应用注册后，下一步是通过 External OAuth 安全集成将该应用注册与 Snowflake 关联起来。安全集成的 `external_oauth_audience_list` 参数必须与你在配置 Azure Entra ID 时指定的 **Application ID URI** 一致。 

**Issuer** 和 **JWS Keys endpoint** 同样来自前面步骤中收集的值。**User Mapping Attribute** 可以设置为 `EMAIL_ADDRESS` 或 `LOGIN_NAME`，用户的 Microsoft 登录凭据将据此映射到其在 Snowflake 中的用户，以确保签发给 ChatGPT 的访问令牌遵循 Snowflake 中的权限。 

In [ ]:
CREATE OR REPLACE SECURITY INTEGRATION AZURE_OAUTH_INTEGRATION
  TYPE = EXTERNAL_OAUTH
  ENABLED = TRUE
  EXTERNAL_OAUTH_TYPE = 'AZURE'
  EXTERNAL_OAUTH_ISSUER = '<AZURE_AD_ISSUER>'
  EXTERNAL_OAUTH_JWS_KEYS_URL = '<AZURE_AD_JWS_KEY_ENDPOINT>'
  EXTERNAL_OAUTH_AUDIENCE_LIST = ('<SNOWFLAKE_APPLICATION_ID_URI>')
  EXTERNAL_OAUTH_TOKEN_USER_MAPPING_CLAIM = 'upn'
  EXTERNAL_OAUTH_SNOWFLAKE_USER_MAPPING_ATTRIBUTE = 'EMAIL_ADDRESS';

### 中间件信息：

请确保在你的 Azure 环境中完成以下步骤：

* 拥有可以创建 Azure Function Apps 和 Azure Entra 应用注册的 Azure 门户或 VS Code 访问权限
* 本指南中有[一个详细的章节](#azure-function-app)，介绍如何部署和设计所需的函数，用于封装来自 Snowflake 的响应，从而将查询结果以 CSV 形式返回给 ChatGPT。Azure Function App 让你的 GPT 能够摄取更大的数据集，因为 ChatGPT 从文件响应中能够摄取的数据比从 application/json 载荷中更多。此外，只有当响应被格式化为 CSV 文件时，这些数据集才能用于数据分析（又称 Code Interpreter）。

### Azure Function App

现在我们已经创建好 GPT 并处理好了 Azure/Snowflake 身份验证，接下来可以创建 Azure Function App 本身，用来执行 SQL 查询并处理响应格式，使 GPT 能够将结果下载为 CSV 以用于数据分析。

关于部署 Azure Function App 的更多详细信息，请参阅这篇 [Azure Cookbook 指南](https://cookbook.openai.com/examples/azure/functions)。下面提供了可添加到该函数中的示例代码。

这段代码仅供参考方向——虽然它应该可以开箱即用，但你应根据你的 GPT 以及 IT 环境的具体需求对其进行定制。

### 应用代码

你需要在 Azure Function App 中设置以下流程：

* 从 HTTP 请求中提取令牌，并用它连接到 Snowflake
* 执行 SQL 查询并将结果写入 CSV
* 将该 CSV 临时存储在 Blob Storage 中*
* 生成一个预签名 URL，用于安全地访问该 CSV*
* 以 openaiFileResponse 作为响应

*如果你使用 [file stream](https://platform.openai.com/docs/actions/getting-started/inline-option) 选项而不是 [url](https://platform.openai.com/docs/actions/getting-started/url-option) 选项向 GPT 返回文件，则可能不需要这些步骤。下文会对此做更多说明。

确保已在脚本中安装并导入所需的库。除 Python 标准库之外，这个示例脚本还用到了以下库：

In [ ]:
import azure.functions as func
from azure.storage.blob import BlobServiceClient, generate_blob_sas, BlobSasPermissions, ContentSettings
import snowflake.connector
import jwt    # pyjwt for token decoding

#### 连接到 Snowflake

要连接到 Snowflake，你需要从 Authorization 请求头中提取 Azure Entra ID 签发的访问令牌，并在连接 Snowflake 服务器时使用该令牌。 

在本示例中，Snowflake 用户名就是电子邮件地址，这简化了从 HTTP 访问令牌中提取的 Entra ID 用户与连接所需的 Snowflake 用户 ID 之间的映射。如果你的组织并非如此，可以在 Python 应用中将电子邮件地址映射到 Snowflake 用户 ID。

我的应用被设计为与单个 Snowflake 账户（例如 ab12345.eastus2.azure）和单个 Warehouse 交互。如果你需要访问多个账户或 warehouse，可以考虑在 GPT action 参数中传入这些参数，以便从 HTTP 请求中提取它们。

In [ ]:
# Extract the token from the Authorization header
auth_header = req.headers.get('Authorization')
token_type, token = auth_header.split()

try:
    # Extract email address from token to use for Snowflake user mapping
    # If Snowflake usernames are not emails, then identify the username accordingly
    decoded_token = jwt.decode(token, options={"verify_signature": False})
    email = decoded_token.get('upn') 
    
    conn = snowflake.connector.connect(
        user=email, # Snowflake username, i.e., user's email in my example
        account=SNOWFLAKE_ACCOUNT, # Snowflake account, i.e., ab12345.eastus2.azure
        authenticator="oauth",
        token=token,
        warehouse=SNOWFLAKE_WAREHOUSE # Replace with Snowflake warehouse
    )
    logging.info("Successfully connected to Snowflake.")
except Exception as e:
    logging.error(f"Failed to connect to Snowflake: {e}")

#### 执行查询并保存 CSV

连接到 Snowflake 后，你需要执行查询并将结果存储为 CSV。虽然 Snowflake 中的角色应该可以杜绝任何有害查询的可能，但你可能仍希望在应用中对查询进行清理（下面未包含），就像对待任何其他以编程方式执行的 SQL 查询一样。 

In [ ]:
# Extract SQL query from request parameters or body
sql_query = req.params.get('sql_query')

try:
    # Use the specified warehouse
    cursor = conn.cursor()

    # Execute the query
    cursor.execute(sql_query)
    results = cursor.fetchall()
    column_names = [desc[0] for desc in cursor.description]
    logger.info(f"Query executed successfully: {sql_query}")

    # Convert results to CSV
    csv_file_path = write_results_to_csv(results, column_names)
except Exception as e:
    logger.error(f"Error executing query or processing data: {e}")


def write_results_to_csv(results, column_names):
    try:
        # Create a temporary file
        temp_file = tempfile.NamedTemporaryFile(delete=False, mode='w', newline='')
        csv_writer = csv.writer(temp_file)
        csv_writer.writerow(column_names)  # Write the column headers
        csv_writer.writerows(results)      # Write the data rows
        temp_file.close()  # Close the file to flush the contents
        return temp_file.name  # Return file path
    except Exception as e:
        logger.error(f"Error writing results to CSV: {e}")

#### 将文件存储到 Blob Storage

有 2 种方法可以将文件返回给 ChatGPT 进行处理。你可以在 openaiFileResponse 列表响应中[流式传输](https://platform.openai.com/docs/actions/getting-started/inline-option) base64 编码的数据以及 mimeType 和文件名，也可以返回一个 [URL 列表](https://platform.openai.com/docs/actions/getting-started/url-option)。在本方案中，我们将重点介绍后者。

为此，你需要将 CSV 上传到 Azure Blob Storage，并返回一个预签名 URL，以便在 ChatGPT 中安全地访问该文件。需要注意的是，为了能在 ChatGPT 中下载某个 URL，你需要确保该 URL 包含 content_type 和 content_disposition，如下面的示例所示。如果想检查某个 URL 是否带有所需的请求头，可以在任意终端中使用 ``curl -I <url>``。

你需要按照[这里](https://learn.microsoft.com/en-us/azure/storage/common/storage-configure-connection-string)的说明，获取 Azure 存储桶的连接字符串。 

In [ ]:
def upload_csv_to_azure(file_path, container_name, blob_name, connect_str):
    try:
        # Create the BlobServiceClient object which will be used to create a container client
        blob_service_client = BlobServiceClient.from_connection_string(connect_str)
        
        # Create a blob client using the local file name as the name for the blob
        blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)

        # Upload the file with specified content settings
        with open(file_path, "rb") as data:
            blob_client.upload_blob(data, overwrite=True, content_settings=ContentSettings(
                content_type='text/csv',
                content_disposition=f'attachment; filename="{blob_name}"'
            ))
        logger.info(f"Successfully uploaded {file_path} to {container_name}/{blob_name}")

        # Generate a SAS token for the blob
        sas_token = generate_blob_sas(
            account_name=blob_service_client.account_name,
            container_name=container_name,
            blob_name=blob_name,
            account_key=blob_service_client.credential.account_key,
            permission=BlobSasPermissions(read=True),
            expiry=datetime.datetime.utcnow() + datetime.timedelta(hours=1)  # Token valid for 1 hour
        )

        # Generate a presigned URL using the SAS token
        url = f"https://{blob_service_client.account_name}.blob.core.windows.net/{container_name}/{blob_name}?{sas_token}"
        logger.info(f"Generated presigned URL: {url}")

        return url
    except Exception as e:
        logger.error(f"Error uploading file to Azure Blob Storage: {e}")
        raise

#### 格式化 openaiFileResponse

最后，你需要适当地格式化响应，指示 ChatGPT 将该响应作为一个或一系列文件进行处理。openaiFileResponse 是一个列表，最多可以包含 10 个 URL（如果使用 [inline 选项](https://platform.openai.com/docs/actions/getting-started/inline-option)，则为 base64 编码内容）。

In [ ]:
# Format the response so ChatGPT treats it as a file
response = {
    'openaiFileResponse': [csv_url]
}
cursor.close()
conn.close()
return func.HttpResponse(
    json.dumps(response), 
    status_code=200
)


这个应用涉及许多环节，因此测试 Azure Function App 可能很重要。ChatGPT 可能不是一个理想的测试环境，因为其请求和响应有时不够透明，不便于调试。先通过 cURL 或 Postman 在更可控的环境中发起 HTTP 请求来对应用进行初步测试，可以让你更轻松地调试和排查问题。一旦确认在这些工具中响应能按预期返回，你就可以着手构建 GPT 了。

## ChatGPT 步骤

### 自定义 GPT 指令

创建自定义 GPT 后，可以参考下面的文本填写 Instructions 面板。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步的工作原理。

##### 指令示例

让 ChatGPT 理解你的表 schema 对于正确构造 SQL 查询非常重要。实现这一点有多种方法，这套指令代表了最直接的方式。我们正在努力发布更多指令，适用于你可能想构建的不同版本的 Snowflake GPT，例如支持处理多个不同的表、schema 和数据库，甚至能够动态学习随时间变化的 schema。

下面是针对单个 schema 和单个表的一些基本指令。这个 GPT 针对单一用例（分析 2013 年 1 月从纽约出发的航班数据）进行了优化，从而可以用最简单的指令获得最可靠的 GPT 表现。

你是编写 SQL 查询以从 Snowflake 获取数据的专家。你帮助用户将他们的提示转换为 SQL 查询。任何关于航班数据的问题都将被转换为查询表 `FLIGHTS.PUBLIC.JAN_2013_NYC` 的 Snowflake SQL 查询。将查询传入 "sql_query" 参数


该表的 schema 包括
```
ID	NUMBER	A unique identifier for each flight
YEAR	NUMBER	The year of the flight
MONTH	NUMBER	The month of the flight
DAY		NUMBER	The day of the month on which the flight departed
DEP_TIME	NUMBER	The actual departure time of the flight
SCHED_DEP_TIME	NUMBER	The scheduled departure time of the flight
DEP_DELAY	NUMBER	The departure delay in minutes (negative values indicate early departures)
ARR_TIME	NUMBER	The actual arrival time of the flight
SCHED_ARR_TIME	NUMBER	The scheduled arrival time of the flight
ARR_DELAY	NUMBER	The arrival delay in minutes (negative values indicate early arrivals)
CARRIER_CODE	TEXT	The carrier code of the airline
FLIGHT	NUMBER	The flight number
TAILNUM	TEXT	The aircraft tail number
ORIGIN_AIRPORT_CODE	TEXT	The origin airport code
DEST_AIRPORT_CODE	TEXT	The destination airport code
AIR_TIME	NUMBER	The total airtime of the flight in minutes
DISTANCE	NUMBER	The distance traveled by the flight in miles
HOUR	NUMBER	The hour part of the scheduled departure time
MINUTE	NUMBER	The minute part of the scheduled departure time
TIME_HOUR	NUMBER	The time at which the flight departed (rounded to the nearest hour)
CARRIER_NAME	TEXT	The full name of the airline carrier
ORIGIN_AIRPORT_NAME	TEXT	The full name of the origin airport
ORIGIN_REGION	TEXT	The region code of the origin airport
ORIGIN_MUNICIPALITY	TEXT	The city where the origin airport is located
ORIGIN_COORDINATES	TEXT	The geographical coordinates of the origin airport
DEST_AIRPORT_NAME	TEXT	The full name of the destination airport
DEST_REGION	TEXT	The region code of the destination airport
DEST_MUNICIPALITY	TEXT	The city where the destination airport is located
DEST_COORDINATES	TEXT	The geographical coordinates of the destination airport
```


当用户询问有关航班的数据时，请执行以下操作：


1. 使用 `executeSQL` action 向 Azure function 端点发送 POST 请求
2. 接收作为 Action 响应一部分返回的文件。将其显示为电子表格
3. 对该文件进行分析，并提供用户所询问的必要信息


用户会希望在 code interpreter 中询问有关数据的问题，因此对于从你拉取的数据集中得出的任何数据分析洞察，都请使用它。

### OpenAPI Schema

创建自定义 GPT 后，将下面的文本复制到 Actions 面板中，用你的函数的具体信息替换占位值，并根据你在 Azure Function App 中添加的任何额外输入更新参数。 

有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步的工作原理。

In [ ]:
openapi: 3.1.0
info:
  title: Snowflake GPT API
  description: API to execute SQL queries on Snowflake and get the results as a CSV file URL.
  version: 1.0.0
servers:
  - url: https://<server-name>.azurewebsites.net
    description: Azure Function App server running Snowflake integration application
paths:
  /api/<function_name>?code=<code>:
    post:
      operationId: executeSQL
      summary: Executes a SQL query on Snowflake and returns the result file URL as a CSV.
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                sql_query:
                  type: string
                  description: The SQL query to be executed on Snowflake.
              required:
                - sql_query
      responses:
        '200':
          description: Successfully executed the query.
          content:
            application/json:
              schema:
                type: object
                properties:
                  openaiFileResponse:
                    type: array
                    items:
                      type: string
                      format: uri
                    description: Array of URLs pointing to the result files.
        '401':
          description: Unauthorized. Missing or invalid authentication token.
        '400':
          description: Bad Request. The request was invalid or cannot be otherwise served.
        '500':
          description: Internal Server Error. An error occurred on the server.
components:
  schemas: {} 

### 常见问题与故障排除

* 返回给 ChatGPT 的文件大小限制为 10MB。如果返回的文件超过该大小，请求可能会失败。如果你发现遇到了这些限制，请务必在 SQL 命令中加入 LIMIT。
* _为什么一开始就需要 Azure Function App？_ ChatGPT 的数据分析功能（又称 Code Interpreter）依赖于一个与模型上下文窗口相隔离的安全 Python 环境。目前，传递给数据分析的数据必须通过上传文件的方式提供。因此，返回数据的 GPT actions 必须以 CSV 或其他数据文件类型返回这些数据。要通过 GPT action 返回文件，响应必须封装在 `openaiFileResponse` 对象中。这需要自定义代码来正确格式化响应。
* _我的公司使用的是 Azure 以外的云服务商。_ 关于通过 GPT action 将其他中间件函数连接到 ChatGPT，请参阅其他 [AWS](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_middleware_aws_function) 或 [GCP](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_middleware_google_cloud_function) 中间件 cookbook。你可以借鉴本 cookbook 中讨论的概念，作为构建中间件应用时的考量参考，但对于不同的云服务商，将中间件连接到 Snowflake 的方式可能有所不同。例如，Snowflake 专门构建了一个用于与 Azure Entra ID 关联的 External OAuth 集成。 
* _如何限制 GPT 可以访问的数据集？_ 限制 ChatGPT 在 Snowflake 中的访问范围可能很重要。有以下几种方法：
    * Snowflake 角色可以限制谁能访问哪些表，并且 Azure Entra ID 为 GPT 用户签发的访问令牌也会遵循这些限制
    * 在中间件函数中，你可以添加合理性检查，验证所访问的表是否已获该应用批准
    * 你可能希望专门为与 ChatGPT 集成生成一个全新的 Database/Warehouse，并清除其中任何敏感信息，例如 PII。
* _Schema 调用了错误的 warehouse 或数据集：_ 如果 ChatGPT 调用了错误的 warehouse 或数据库，可以考虑更新指令，使其更明确地 (a) 指出应调用哪个 warehouse / 数据库，或者 (b) 要求用户在运行查询之前提供这些确切信息


_有没有希望我们优先支持的集成？我们的集成中有错误吗？请在我们的 GitHub 中提交 PR 或 issue，我们会尽快查看。_